# ☕ Coffee Shop EDA — Matplotlib, Seaborn & Plotly

This notebook explores coffee shop transaction data using **three** visualisation libraries side-by-side:

| Library | Best For |
|---|---|
| **Matplotlib** | Full control, foundational charts |
| **Seaborn** | Beautiful statistical charts with less code |
| **Plotly** | Interactive charts (hover, zoom, click) |

---
### Quick Reference — Which Chart for Which Question?

| Business Question | Best Chart |
|---|---|
| Compare categories | Bar Chart |
| Rank categories | Horizontal Bar Chart |
| Show percentages of a whole | Pie Chart |
| Show trends over time | Line Chart |
| Count occurrences | Count Plot |
| Show distributions | Histogram |
| Show spread and outliers | Box Plot |
| Show relationships between variables | Heatmap |

### Easy Rule to Remember
```
Bar       = Compare
Pie       = Percentage
Line      = Trend
Histogram = Distribution
Box Plot  = Spread
Heatmap   = Relationship
```

---
## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_style('whitegrid')
print('✅ Libraries loaded!')

---
## Load & Inspect the Data

In [ ]:
df = pd.read_csv(
    'coffee_shop_transactions_cleaned.csv',
    parse_dates=['DateTime']   # parse_dates converts the column to a real datetime
                                # without this, DateTime loads as plain text and
                                # .dt.date / .dt.hour etc. would not work
)

# Extract useful date/time parts for later charts
df['Date'] = df['DateTime'].dt.date
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.day_name()

print(f"Shape: {df.shape[0]} transactions, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

---
## 1️⃣ Bar Chart — Compare Categories
**Question:** Which items generate the most total revenue?

In [ ]:
revenue_by_item = df.groupby('Item')['TotalPrice'].sum().sort_values(ascending=False)
revenue_by_item

### Matplotlib

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(revenue_by_item.index, revenue_by_item.values, color='saddlebrown')
plt.xlabel('Item')
plt.ylabel('Total Revenue ($)')
plt.title('Revenue by Item — Matplotlib')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Seaborn

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=revenue_by_item.index, y=revenue_by_item.values, hue=revenue_by_item.index,
            palette='copper_r', legend=False)
plt.xlabel('Item')
plt.ylabel('Total Revenue ($)')
plt.title('Revenue by Item — Seaborn')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Plotly (Interactive — hover over bars!)

In [ ]:
fig = px.bar(
    x=revenue_by_item.index, y=revenue_by_item.values,
    labels={'x': 'Item', 'y': 'Total Revenue ($)'},
    title='Revenue by Item — Plotly',
    color=revenue_by_item.values, color_continuous_scale='Brwnyl'
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

---
## 2️⃣ Horizontal Bar Chart — Rank Categories
**Question:** Which items sold the most units? (Ranked)

In [ ]:
qty_by_item = df.groupby('Item')['Quantity'].sum().sort_values()
qty_by_item

### Matplotlib

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(qty_by_item.index, qty_by_item.values, color='peru')
plt.xlabel('Units Sold')
plt.ylabel('Item')
plt.title('Items Ranked by Units Sold — Matplotlib')
plt.tight_layout()
plt.show()

### Seaborn

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=qty_by_item.values, y=qty_by_item.index, hue=qty_by_item.index,
            palette='YlOrBr', legend=False, orient='h')
plt.xlabel('Units Sold')
plt.ylabel('Item')
plt.title('Items Ranked by Units Sold — Seaborn')
plt.tight_layout()
plt.show()

### Plotly (Interactive)

In [ ]:
fig = px.bar(
    x=qty_by_item.values, y=qty_by_item.index, orientation='h',
    labels={'x': 'Units Sold', 'y': 'Item'},
    title='Items Ranked by Units Sold — Plotly',
    color=qty_by_item.values, color_continuous_scale='Oranges'
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

---
## 3️⃣ Pie Chart — Show Percentages of a Whole
**Question:** What share of total revenue comes from each payment method?

In [ ]:
revenue_by_payment = df.groupby('PaymentMethod')['TotalPrice'].sum()
revenue_by_payment

### Matplotlib

In [ ]:
plt.figure(figsize=(6, 6))
plt.pie(revenue_by_payment.values, labels=revenue_by_payment.index, autopct='%1.1f%%',
        colors=['#8B4513', '#D2691E', '#CD853F'], startangle=90)
plt.title('Revenue Share by Payment Method — Matplotlib')
plt.tight_layout()
plt.show()

### Seaborn
*Seaborn has no built-in pie chart — it focuses on statistical charts. We use Matplotlib's pie() with a Seaborn colour palette instead.*

In [ ]:
colors = sns.color_palette('YlOrBr', len(revenue_by_payment))

plt.figure(figsize=(6, 6))
plt.pie(revenue_by_payment.values, labels=revenue_by_payment.index, autopct='%1.1f%%',
        colors=colors, startangle=90)
plt.title('Revenue Share by Payment Method — Seaborn Palette')
plt.tight_layout()
plt.show()

### Plotly (Interactive)

In [ ]:
fig = px.pie(
    values=revenue_by_payment.values, names=revenue_by_payment.index,
    title='Revenue Share by Payment Method — Plotly',
    color_discrete_sequence=px.colors.sequential.Brwnyl
)
fig.update_traces(textinfo='percent+label')
fig.show()

---
## 4️⃣ Line Chart — Show Trends Over Time
**Question:** How does daily revenue trend across the month?

In [ ]:
revenue_by_date = df.groupby('Date')['TotalPrice'].sum().reset_index()
revenue_by_date.head()

### Matplotlib

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(revenue_by_date['Date'], revenue_by_date['TotalPrice'],
         marker='o', color='saddlebrown', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Daily Revenue ($)')
plt.title('Daily Revenue Trend — Matplotlib')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Seaborn

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=revenue_by_date, x='Date', y='TotalPrice',
             marker='o', color='peru', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Daily Revenue ($)')
plt.title('Daily Revenue Trend — Seaborn')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Plotly (Interactive — hover to see exact values)

In [ ]:
fig = px.line(
    revenue_by_date, x='Date', y='TotalPrice', markers=True,
    labels={'TotalPrice': 'Daily Revenue ($)'},
    title='Daily Revenue Trend — Plotly'
)
fig.update_traces(line_color='#8B4513')
fig.show()

---
## 5️⃣ Count Plot — Count Occurrences
**Question:** How many transactions were made with each payment method?

### Matplotlib

In [ ]:
counts = df['PaymentMethod'].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(counts.index, counts.values, color='chocolate')
plt.xlabel('Payment Method')
plt.ylabel('Number of Transactions')
plt.title('Transaction Count by Payment Method — Matplotlib')
plt.tight_layout()
plt.show()

### Seaborn
*Seaborn has a dedicated `countplot()` — it counts rows automatically, no need to pre-aggregate!*

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x='PaymentMethod', hue='PaymentMethod',
              palette='copper_r', legend=False,
              order=df['PaymentMethod'].value_counts().index)
plt.xlabel('Payment Method')
plt.ylabel('Number of Transactions')
plt.title('Transaction Count by Payment Method — Seaborn')
plt.tight_layout()
plt.show()

### Plotly (Interactive)

In [ ]:
fig = px.histogram(
    df, x='PaymentMethod',
    title='Transaction Count by Payment Method — Plotly',
    color='PaymentMethod', color_discrete_sequence=px.colors.sequential.Brwnyl
)
fig.update_layout(showlegend=False, yaxis_title='Number of Transactions')
fig.show()

---
## 6️⃣ Histogram — Show Distributions
**Question:** What does the distribution of transaction amounts look like?

### Matplotlib

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['TotalPrice'], bins=8, color='sienna', edgecolor='white')
plt.xlabel('Total Price ($)')
plt.ylabel('Frequency')
plt.title('Distribution of Transaction Amounts — Matplotlib')
plt.tight_layout()
plt.show()

### Seaborn

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['TotalPrice'], bins=8, color='peru', kde=True)
plt.xlabel('Total Price ($)')
plt.ylabel('Frequency')
plt.title('Distribution of Transaction Amounts — Seaborn (with KDE curve)')
plt.tight_layout()
plt.show()

### Plotly (Interactive)

In [ ]:
fig = px.histogram(
    df, x='TotalPrice', nbins=8,
    title='Distribution of Transaction Amounts — Plotly',
    color_discrete_sequence=['#A0522D']
)
fig.update_layout(yaxis_title='Frequency')
fig.show()

---
## 7️⃣ Box Plot — Show Spread and Outliers
**Question:** How does the price per item vary across different items?

### Matplotlib

In [ ]:
items = df['Item'].unique()
data_by_item = [df[df['Item'] == item]['TotalPrice'].values for item in items]

plt.figure(figsize=(10, 5))
plt.boxplot(data_by_item, tick_labels=items, patch_artist=True,
            boxprops=dict(facecolor='burlywood'))
plt.xlabel('Item')
plt.ylabel('Total Price ($)')
plt.title('Price Spread by Item — Matplotlib')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Seaborn

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Item', y='TotalPrice', hue='Item', palette='copper_r', legend=False)
plt.xlabel('Item')
plt.ylabel('Total Price ($)')
plt.title('Price Spread by Item — Seaborn')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Plotly (Interactive — hover to see min/max/median)

In [ ]:
fig = px.box(
    df, x='Item', y='TotalPrice',
    title='Price Spread by Item — Plotly',
    color='Item', color_discrete_sequence=px.colors.sequential.Brwnyl
)
fig.update_layout(showlegend=False)
fig.show()

---
## 8️⃣ Heatmap — Show Relationships Between Variables
**Question:** How are PricePerItem, Quantity, and TotalPrice correlated?

In [ ]:
corr_matrix = df[['PricePerItem', 'Quantity', 'TotalPrice', 'Hour']].corr()
corr_matrix.round(2)

### Matplotlib

In [ ]:
plt.figure(figsize=(6, 5))
im = plt.imshow(corr_matrix, cmap='copper', vmin=-1, vmax=1)
plt.colorbar(im, label='Correlation')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)

# Annotate each cell with its value
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        plt.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', color='white')

plt.title('Correlation Heatmap — Matplotlib')
plt.tight_layout()
plt.show()

### Seaborn
*Seaborn's `heatmap()` adds annotations and colour scaling automatically — much less code!*

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='copper', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Heatmap — Seaborn')
plt.tight_layout()
plt.show()

### Plotly (Interactive — hover over each cell)

In [ ]:
fig = px.imshow(
    corr_matrix, text_auto='.2f', color_continuous_scale='Brwnyl',
    zmin=-1, zmax=1, title='Correlation Heatmap — Plotly'
)
fig.show()

---
## Summary

| # | Chart | Question Answered | Key Insight Column(s) |
|---|---|---|---|
| 1 | Bar Chart | Compare categories | `Item` vs `TotalPrice` |
| 2 | Horizontal Bar | Rank categories | `Item` vs `Quantity` |
| 3 | Pie Chart | % of a whole | `PaymentMethod` vs `TotalPrice` |
| 4 | Line Chart | Trend over time | `Date` vs `TotalPrice` |
| 5 | Count Plot | Count occurrences | `PaymentMethod` |
| 6 | Histogram | Show distribution | `TotalPrice` |
| 7 | Box Plot | Spread & outliers | `Item` vs `TotalPrice` |
| 8 | Heatmap | Relationships | `PricePerItem`, `Quantity`, `TotalPrice`, `Hour` |

### When to use which library?
- **Matplotlib** → maximum control, customise every pixel
- **Seaborn** → faster to write, great statistical defaults (e.g. built-in `countplot`, KDE curves)
- **Plotly** → interactive dashboards, hover tooltips, presentations & stakeholders